# ARG Protein Feature Extraction — Pfeature (batch, per gene family)

Computes composition-based Pfeature descriptors for every family FASTA in `INPUT_DIR`, writes one merged CSV per family to `OUTPUT_DIR`, plus a combined table across all families.

**Scope (tested directly against pfeature v1.4, not just going by the PDF manual):**
- Jobs run: `AAC, DPC, PCP, ATC, BTC, RRI, PRI, DDR, SEP, SER, SPC, CTC, PAAC, APAAC, QSO, SOC`
- Excluded — `TPC`, `AAI`, `ACR`: didn't finish in 25s on 5 short (~300aa) sequences, not viable at family scale as-is
- Excluded — `CeTD`, `ALLCOMP`: crash on pandas ≥2.0 (`DataFrame.append` was removed; the package still calls it)
- Not available via `pip install pfeature` at all: PSSM/evolutionary info (needs PSI-BLAST + nr/swissprot), structure descriptors (needs PDB files + PaDEL/NACCESS/DSSP/openbabel), Pattern module — those are Sections 3–5 of the manual and require the separate GitHub PyLib/Standalone downloads plus external tools, not this pip install.

**Requires:** `pip install pfeature tqdm` (tqdm is used internally by pfeature_comp but isn't declared as a dependency, so it has to be installed separately or the CLI throws `ModuleNotFoundError`)

**Before running:** set `INPUT_DIR` below to your per-family protein FASTA directory.

In [1]:
!pfeature_comp -h

usage: pfeature_comp [-h] -i INPUT [-o OUTPUT]
                     [-j {AAC,DPC,TPC,ATC,BTC,PCP,AAI,RRI,PRI,DDR,SEP,SER,SPC,ACR,CTC,CeTD,PAAC,APAAC,QSO,SOC,ALLCOMP}]
                     [-n N_TERMINAL] [-c C_TERMINAL] [-nct NC_TERMINAL]
                     [-rn REST_N] [-rc REST_C] [-rnc REST_NC] [-s SPLIT]
                     [-d LAG] [-w WEIGHT] [-t PWEIGHT]

Please provide following arguments

options:
  -h, --help            show this help message and exit
  -i INPUT, -I INPUT, --input INPUT
                        Input: protein or peptide sequence in FASTA format or single sequence per line in single letter code
  -o OUTPUT, -O OUTPUT, --output OUTPUT
                        Output: File for saving results by default pfeature_result.csv
  -j {AAC,DPC,TPC,ATC,BTC,PCP,AAI,RRI,PRI,DDR,SEP,SER,SPC,ACR,CTC,CeTD,PAAC,APAAC,QSO,SOC,ALLCOMP}, -J {AAC,DPC,TPC,ATC,BTC,PCP,AAI,RRI,PRI,DDR,SEP,SER,SPC,ACR,CTC,CeTD,PAAC,APAAC,QSO,SOC,ALLCOMP}, --job {AAC,DPC,TPC,ATC,BTC,PCP,AAI,RRI,PRI,DD

In [2]:
!pip install pfeature tqdm

In [1]:
import os, subprocess, shutil, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm

found = shutil.which("pfeature_comp")
print(f"pfeature_comp: {'OK -> ' + found if found else 'NOT FOUND -- run: pip install pfeature tqdm'}")
print(f"pandas {pd.__version__}")

pfeature_comp: OK -> /home/user/miniforge3/envs/jupyter/bin/pfeature_comp
pandas 2.3.3


/home/user/miniforge3/envs/jupyter/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- point this at your per-family protein FASTA directory ---
INPUT_DIR = Path("/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas/protein")

OUTPUT_DIR = Path("/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas_pfeatures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FASTA_GLOBS = ["*.fasta", "*.fa", "*.faa"]

# Confirmed working + reasonably fast on pfeature v1.4 (tested directly: 5 seqs, ~300aa each, all <3.1s).
# Excluded: TPC, AAI, ACR   -- didn't finish in 25s for 5 short sequences
#           CeTD, ALLCOMP   -- crash on pandas>=2.0 (DataFrame.append removed)
JOBS = ["AAC", "DPC", "PCP", "ATC", "BTC", "RRI", "PRI", "DDR",
        "SEP", "SER", "SPC", "CTC", "PAAC", "APAAC", "QSO", "SOC"]

LAG = 1        # order/lag/lambda shared by DPC, QSO, SOC, PAAC, APAAC (pfeature_comp's -d flag, default 1)
N_WORKERS = 14  # set to your actual allocated cores -- os.cpu_count() reports the whole node, not your job's share

### Notes

- Pfeature's raw CSV output has **no sequence-ID column** — row order matches FASTA order exactly (verified directly: tested with distinct poly-A/poly-G/poly-W sequences and multi-line-wrapped FASTA, in that order, and confirmed the output rows came back in the same order). IDs are reattached from the FASTA headers after the fact below — don't reorder or filter rows between running a job and merging.
- Composition modules assume the 20 standard amino acids. Sequences with `X`/`B`/`Z`/`U`/`*` or gaps (e.g. from partial/degenerate CDS) are flagged by the pre-flight scan below but not excluded automatically — given the known CARD reference-quality issues in this dataset (e.g. the mef(H) broken partial entry), that's a judgment call for you, not something to silently drop.
- **Concurrency bug in the package itself, found by testing this exact batch pattern:** every `pfeature_comp` job writes its intermediate result to a hardcoded relative scratch filename (`tempfile_out`, `sam_input.csv`) in the process's current working directory — not a unique per-run name, despite the package importing `uuid`. Run two jobs at once from the same working directory and they silently step on each other's scratch files: no error, no non-zero exit code, just missing or wrong output. Caught this directly — a `DPC.csv` for one family went missing under 4-way parallelism with zero failures reported. Fix used below: each subprocess call runs in its own isolated temp working directory, so the relative scratch paths never collide, regardless of how many workers run at once.

In [5]:
import tempfile, shutil as sh

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

def discover_family_fastas(input_dir: Path):
    files = set()
    for pattern in FASTA_GLOBS:
        files.update(input_dir.glob(pattern))
    return sorted(files)

def parse_fasta_ids(fasta_path: Path):
    ids = []
    with open(fasta_path) as fh:
        for line in fh:
            if line.startswith(">"):
                ids.append(line[1:].strip().split()[0])
    return ids

def scan_nonstandard_residues(fasta_path: Path):
    flagged = {}
    header, seq_chars = None, set()
    def _check():
        extra = seq_chars - STANDARD_AA
        if extra:
            flagged[header] = sorted(extra)
    with open(fasta_path) as fh:
        for line in fh:
            line = line.strip()
            if line.startswith(">"):
                if header is not None:
                    _check()
                header, seq_chars = line[1:].split()[0], set()
            else:
                seq_chars.update(line.upper())
        if header is not None:
            _check()
    return flagged

def run_pfeature_job(fasta_path: Path, job: str, out_csv: Path):
    # pfeature_comp writes scratch files (tempfile_out, sam_input.csv) to a HARDCODED
    # relative path in its cwd for every job type -- concurrent processes sharing a
    # cwd can silently clobber each other's scratch files (confirmed: no error, no
    # bad exit code, output just goes missing). Isolating cwd per call sidesteps it.
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    scratch_dir = tempfile.mkdtemp(prefix="pfeature_")
    try:
        cmd = ["pfeature_comp", "-i", str(fasta_path.resolve()), "-j", job,
               "-o", str(out_csv.resolve()), "-d", str(LAG)]
        result = subprocess.run(cmd, cwd=scratch_dir, capture_output=True, text=True)
        ok = result.returncode == 0 and out_csv.exists()
        return ok, result.stderr
    finally:
        sh.rmtree(scratch_dir, ignore_errors=True)

In [6]:
family_fastas = discover_family_fastas(INPUT_DIR)
print(f"Found {len(family_fastas)} family FASTA file(s) in {INPUT_DIR}")

nonstandard_report = {}
for fp in family_fastas:
    flagged = scan_nonstandard_residues(fp)
    if flagged:
        nonstandard_report[fp.name] = flagged

if nonstandard_report:
    print(f"\n{len(nonstandard_report)} file(s) contain non-standard residues (X/B/Z/U/*/gaps) -- not blocking, just flagging:")
    for fname, seqs in nonstandard_report.items():
        print(f"  {fname}: {len(seqs)} sequence(s) affected, e.g. {list(seqs.items())[:2]}")
else:
    print("No non-standard residues detected.")

Found 39 family FASTA file(s) in /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas/protein

3 file(s) contain non-standard residues (X/B/Z/U/*/gaps) -- not blocking, just flagging:
  cephalosporin.fasta: 2 sequence(s) affected, e.g. [('ARO:3004647|gb|AHF82023.1|ARO:3004647|AQU-2', ['X']), ('ARO:3004648|gb|AHF82024.1|ARO:3004648|AQU-3', ['X'])]
  rifamycin_antibiotic.fasta: 1 sequence(s) affected, e.g. [('ARO:3000444|gb|AIA08936.1|ARO:3000444|rphA', ['X'])]
  tetracycline_antibiotic.fasta: 2 sequence(s) affected, e.g. [('ARO:3000573|gb|ACS83748.1|ARO:3000573|tet(43)', ['X']), ('ARO:3002894|gb|AAR96051.1|ARO:3002894|otr(C)', ['X'])]


In [7]:
tasks = [(fp, job) for fp in family_fastas for job in JOBS]
print(f"Running {len(tasks)} (family x job) combinations across {N_WORKERS} workers...")

failures = []

def _worker(task):
    fp, job = task
    out_csv = OUTPUT_DIR / fp.stem / f"{job}.csv"
    ok, err = run_pfeature_job(fp, job, out_csv)
    return fp.name, job, ok, err

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(_worker, t) for t in tasks]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        fname, job, ok, err = fut.result()
        if not ok:
            failures.append((fname, job, err.strip()[-300:]))

print(f"\nDone. {len(tasks) - len(failures)}/{len(tasks)} succeeded.")
if failures:
    print(f"{len(failures)} failure(s):")
    for fname, job, err in failures[:20]:
        print(f"  {fname} / {job}: {err}")

Running 624 (family x job) combinations across 14 workers...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 621/624 [15:43:54<04:33, 91.20s/it]


KeyboardInterrupt: 

In [8]:
merge_failures = []

for fp in tqdm(family_fastas, desc="merging"):
    ids = parse_fasta_ids(fp)
    job_dir = OUTPUT_DIR / fp.stem
    job_frames = []
    for job in JOBS:
        job_csv = job_dir / f"{job}.csv"
        if not job_csv.exists():
            continue
        df = pd.read_csv(job_csv)
        if len(df) != len(ids):
            merge_failures.append((fp.name, job, f"{len(df)} rows vs {len(ids)} sequence IDs -- SKIPPED, row alignment not trustworthy"))
            continue
        job_frames.append(df)
    if not job_frames:
        merge_failures.append((fp.name, "ALL", "no job outputs available to merge"))
        continue
    merged = pd.concat([pd.Series(ids, name="sequence_id")] + job_frames, axis=1)
    merged.to_csv(OUTPUT_DIR / f"{fp.stem}_features.csv", index=False)

print(f"Merged per-family feature tables written to {OUTPUT_DIR}/<family>_features.csv")
if merge_failures:
    print(f"\n{len(merge_failures)} merge issue(s):")
    for fname, job, msg in merge_failures[:20]:
        print(f"  {fname} / {job}: {msg}")

merging: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 39/39 [00:07<00:00,  5.31it/s]

Merged per-family feature tables written to /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas_pfeatures/<family>_features.csv


In [6]:
# Combined table across all families (row-bind; same JOBS list -> same columns for every family)
family_tables = sorted(OUTPUT_DIR.glob("*_features.csv"))
if family_tables:
    frames = []
    for ft in family_tables:
        df = pd.read_csv(ft)
        fam_col = pd.Series([ft.stem.replace("_features", "")] * len(df), name="family")
        frames.append(pd.concat([fam_col, df], axis=1))
    master = pd.concat(frames, axis=0, ignore_index=True)
    master.to_csv(OUTPUT_DIR / "all_families_features.csv", index=False)
    print(f"Master table: {master.shape[0]} sequences x {master.shape[1]-2} features across {len(family_tables)} families")
    print(f"-> {OUTPUT_DIR / 'all_families_features.csv'}")
else:
    print("No merged family tables found -- check failures in the cells above.")

Master table: 12913 sequences x 1000 features across 39 families
-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas_pfeatures/all_families_features.csv
